# Preços de Imóveis — Rio de Janeiro
## Pipeline: Simulação · EDA · Feature Engineering · Modelagem · SHAP

Dataset simulado de **15.000 imóveis residenciais** no Rio de Janeiro com preços de venda realistas por bairro.  
Executar as células em ordem para gerar todos os artefatos em `streamlit_data/`.

In [1]:
# ── Bloco 0 · Imports e diretórios ──────────────────────────────────────────
import numpy as np
import pandas as pd
import json
import pickle
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

DIRS = [
    'streamlit_data/eda',
    'streamlit_data/features',
    'streamlit_data/modeling',
    'streamlit_data/interpretation',
    'streamlit_data/models',
]
for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

print('Diretórios prontos.')

Diretórios prontos.


In [2]:
# ── Bloco 1 · Simulação do dataset ──────────────────────────────────────────
N = 15_000
rng = np.random.default_rng(42)

# Bairros do Rio com coordenadas, preço/m² mediano e peso amostral
BAIRROS = {
    'Leblon':           {'lat': -22.9839, 'lon': -43.2238, 'preco_m2': 22000, 'peso': 3},
    'Ipanema':          {'lat': -22.9838, 'lon': -43.2096, 'preco_m2': 20000, 'peso': 4},
    'Copacabana':       {'lat': -22.9711, 'lon': -43.1872, 'preco_m2': 15000, 'peso': 6},
    'Lagoa':            {'lat': -22.9696, 'lon': -43.2117, 'preco_m2': 18000, 'peso': 2},
    'Botafogo':         {'lat': -22.9483, 'lon': -43.1761, 'preco_m2': 13000, 'peso': 4},
    'Flamengo':         {'lat': -22.9342, 'lon': -43.1757, 'preco_m2': 11000, 'peso': 3},
    'Humaita':          {'lat': -22.9580, 'lon': -43.1924, 'preco_m2': 14000, 'peso': 2},
    'Gavea':            {'lat': -22.9765, 'lon': -43.2390, 'preco_m2': 16000, 'peso': 2},
    'Jardim Botanico':  {'lat': -22.9667, 'lon': -43.2256, 'preco_m2': 17000, 'peso': 2},
    'Sao Conrado':      {'lat': -23.0063, 'lon': -43.2736, 'preco_m2': 12000, 'peso': 2},
    'Barra da Tijuca':  {'lat': -23.0004, 'lon': -43.3649, 'preco_m2': 10000, 'peso': 8},
    'Recreio':          {'lat': -23.0186, 'lon': -43.4710, 'preco_m2': 8000,  'peso': 5},
    'Jacarepagua':      {'lat': -22.9607, 'lon': -43.3778, 'preco_m2': 6000,  'peso': 4},
    'Tijuca':           {'lat': -22.9222, 'lon': -43.2375, 'preco_m2': 8000,  'peso': 5},
    'Vila Isabel':      {'lat': -22.9178, 'lon': -43.2614, 'preco_m2': 7000,  'peso': 3},
    'Sao Cristovao':    {'lat': -22.8991, 'lon': -43.2261, 'preco_m2': 6000,  'peso': 3},
    'Meier':            {'lat': -22.9039, 'lon': -43.2843, 'preco_m2': 5500,  'peso': 4},
    'Engenho Novo':     {'lat': -22.9107, 'lon': -43.2783, 'preco_m2': 5000,  'peso': 3},
    'Ramos':            {'lat': -22.8558, 'lon': -43.2551, 'preco_m2': 4500,  'peso': 3},
    'Madureira':        {'lat': -22.8710, 'lon': -43.3385, 'preco_m2': 4000,  'peso': 4},
    'Centro':           {'lat': -22.9068, 'lon': -43.1729, 'preco_m2': 7000,  'peso': 4},
    'Santa Teresa':     {'lat': -22.9209, 'lon': -43.1840, 'preco_m2': 9000,  'peso': 2},
    'Lapa':             {'lat': -22.9122, 'lon': -43.1797, 'preco_m2': 8000,  'peso': 2},
    'Niteroi':          {'lat': -22.8830, 'lon': -43.1033, 'preco_m2': 7500,  'peso': 4},
    'Campo Grande':     {'lat': -22.9022, 'lon': -43.5600, 'preco_m2': 3500,  'peso': 5},
}

bairro_names = list(BAIRROS.keys())
pesos = np.array([BAIRROS[b]['peso'] for b in bairro_names], dtype=float)
pesos_norm = pesos / pesos.sum()

bairro_idx = rng.choice(len(bairro_names), size=N, p=pesos_norm)
bairro_col = np.array(bairro_names)[bairro_idx]

# Tipo de imóvel
tipos_pool = ['Apartamento'] * 6 + ['Casa', 'Cobertura', 'Studio/Loft', 'Flat']
tipo_col = rng.choice(tipos_pool, size=N)

# Área útil (m²)
area_col = np.zeros(N)
for i, t in enumerate(tipo_col):
    if   t == 'Studio/Loft':  area_col[i] = rng.lognormal(np.log(38),  0.25)
    elif t == 'Flat':          area_col[i] = rng.lognormal(np.log(55),  0.25)
    elif t == 'Apartamento':   area_col[i] = rng.lognormal(np.log(90),  0.40)
    elif t == 'Casa':          area_col[i] = rng.lognormal(np.log(200), 0.45)
    elif t == 'Cobertura':     area_col[i] = rng.lognormal(np.log(220), 0.35)
area_col = np.clip(area_col, 18, 900).round(1)

# Quartos (correlacionado com área)
quartos_col = np.zeros(N, dtype=int)
for i, a in enumerate(area_col):
    if   a < 35:   quartos_col[i] = rng.choice([0, 1], p=[0.50, 0.50])
    elif a < 70:   quartos_col[i] = rng.choice([1, 2], p=[0.45, 0.55])
    elif a < 120:  quartos_col[i] = rng.choice([2, 3], p=[0.45, 0.55])
    elif a < 220:  quartos_col[i] = rng.choice([3, 4], p=[0.50, 0.50])
    else:          quartos_col[i] = rng.choice([4, 5], p=[0.50, 0.50])

suites_col   = np.array([int(rng.integers(0, max(1, q))) for q in quartos_col])
banheiros_col = np.array([max(1, q + int(rng.integers(0, 3))) for q in quartos_col])
vagas_col    = np.array([int(rng.integers(0, max(2, min(4, q + 1)))) for q in quartos_col])

andar_col = np.zeros(N, dtype=int)
for i, t in enumerate(tipo_col):
    andar_col[i] = 0 if t == 'Casa' else int(rng.integers(1, 31))

idade_col = np.clip(rng.exponential(15, N), 0, 65).astype(int)

lat_col = np.array([BAIRROS[b]['lat'] + float(rng.normal(0, 0.008)) for b in bairro_col])
lon_col = np.array([BAIRROS[b]['lon'] + float(rng.normal(0, 0.008)) for b in bairro_col])

# Amenidades
PRAIA_BAIRROS = {'Leblon', 'Ipanema', 'Copacabana', 'Lagoa', 'Flamengo',
                 'Barra da Tijuca', 'Recreio', 'Sao Conrado'}

varanda_col      = (rng.random(N) > 0.35).astype(int)
piscina_col      = (rng.random(N) > 0.68).astype(int)
academia_col     = (rng.random(N) > 0.55).astype(int)
portaria_24h_col = (rng.random(N) > 0.45).astype(int)

vista_mar_col = np.zeros(N, dtype=int)
for i in range(N):
    p = 0.28 if bairro_col[i] in PRAIA_BAIRROS else 0.04
    if andar_col[i] > 10:
        p = min(0.75, p * 1.8)
    vista_mar_col[i] = int(rng.random() < p)

# Condomínio e IPTU
condominio_col = np.array([
    max(0.0, float(rng.normal(BAIRROS[b]['preco_m2'] / 9, BAIRROS[b]['preco_m2'] / 30)))
    for b in bairro_col
]).round(2)

iptu_col = np.array([
    max(0.0, float(rng.normal(
        BAIRROS[b]['preco_m2'] * area_col[i] * 0.0008,
        BAIRROS[b]['preco_m2'] * area_col[i] * 0.0002
    )))
    for i, b in enumerate(bairro_col)
]).round(2)

# Preço de venda (função dos atributos + ruído log-normal)
preco_col = np.zeros(N)
for i in range(N):
    b = bairro_col[i]
    preco = BAIRROS[b]['preco_m2'] * area_col[i]
    preco *= (1 + suites_col[i] * 0.05)
    preco *= (1 + max(0, andar_col[i] - 5) * 0.004)
    preco *= max(0.65, 1 - idade_col[i] * 0.005)
    if vista_mar_col[i]:      preco *= float(rng.uniform(1.18, 1.35))
    if varanda_col[i]:        preco *= 1.025
    if piscina_col[i]:        preco *= 1.040
    if academia_col[i]:       preco *= 1.015
    if portaria_24h_col[i]:   preco *= 1.020
    preco *= (1 + vagas_col[i] * 0.025)
    preco *= float(rng.lognormal(0, 0.12))
    preco_col[i] = preco

df = pd.DataFrame({
    'id_imovel':    range(1, N + 1),
    'bairro':       bairro_col,
    'tipo':         tipo_col,
    'area_util':    area_col,
    'quartos':      quartos_col,
    'suites':       suites_col,
    'banheiros':    banheiros_col,
    'vagas':        vagas_col,
    'andar':        andar_col,
    'idade_imovel': idade_col,
    'varanda':      varanda_col,
    'piscina':      piscina_col,
    'academia':     academia_col,
    'portaria_24h': portaria_24h_col,
    'vista_mar':    vista_mar_col,
    'condominio':   condominio_col,
    'iptu':         iptu_col,
    'lat':          lat_col.round(6),
    'lon':          lon_col.round(6),
    'preco_venda':  preco_col.round(2),
})

df.to_csv('imoveis_rj.csv', index=False)
print(f'Dataset gerado: {len(df):,} imóveis | {df.columns.tolist()}')
print(df[['area_util', 'quartos', 'preco_venda']].describe().round(0))

Dataset gerado: 15,000 imóveis | ['id_imovel', 'bairro', 'tipo', 'area_util', 'quartos', 'suites', 'banheiros', 'vagas', 'andar', 'idade_imovel', 'varanda', 'piscina', 'academia', 'portaria_24h', 'vista_mar', 'condominio', 'iptu', 'lat', 'lon', 'preco_venda']
       area_util  quartos  preco_venda
count    15000.0  15000.0      15000.0
mean       114.0      3.0    1316198.0
std         81.0      1.0    1346064.0
min         18.0      0.0      82038.0
25%         59.0      2.0     493761.0
50%         90.0      3.0     897715.0
75%        142.0      3.0    1635996.0
max        900.0      5.0   17265432.0


In [3]:
# ── Bloco 2 · Artefatos de EDA ──────────────────────────────────────────────

# 1. Distribuições de drivers
driver_specs = {
    'area_util':    ('Área Útil (m²)',          'histogram'),
    'quartos':      ('Quartos',                  'bar'),
    'suites':       ('Suítes',                   'bar'),
    'banheiros':    ('Banheiros',                'bar'),
    'vagas':        ('Vagas de Garagem',          'bar'),
    'andar':        ('Andar',                    'histogram'),
    'idade_imovel': ('Idade do Imóvel (anos)',   'histogram'),
    'condominio':   ('Condomínio (R$/mês)',       'histogram'),
    'iptu':         ('IPTU (R$/ano)',            'histogram'),
    'tipo':         ('Tipo de Imóvel',           'bar'),
    'bairro':       ('Bairro',                   'bar'),
    'varanda':      ('Varanda',                  'bar'),
    'piscina':      ('Piscina',                  'bar'),
    'academia':     ('Academia',                 'bar'),
    'portaria_24h': ('Portaria 24h',             'bar'),
    'vista_mar':    ('Vista Mar',                'bar'),
}

driver_dists = {}
for col, (label, kind) in driver_specs.items():
    if kind == 'histogram':
        vals = df[col].dropna().values
        counts, edges = np.histogram(vals, bins=30)
        driver_dists[col] = {
            'type': 'histogram', 'label': label,
            'x': [float((edges[i] + edges[i+1]) / 2) for i in range(len(counts))],
            'y': counts.tolist(),
        }
    else:
        vc = df[col].value_counts().sort_index()
        driver_dists[col] = {
            'type': 'bar', 'label': label,
            'x': [str(v) for v in vc.index.tolist()],
            'y': vc.values.tolist(),
        }

with open('streamlit_data/eda/driver_distributions.json', 'w', encoding='utf-8') as f:
    json.dump(driver_dists, f, ensure_ascii=False)

# 2. Distribuição de preços
p_vals     = df['preco_venda'].values
log_p_vals = np.log1p(p_vals)
c_raw, e_raw = np.histogram(p_vals, bins=50)
c_log, e_log = np.histogram(log_p_vals, bins=50)
price_dist = {
    'raw': {'x': [float((e_raw[i]+e_raw[i+1])/2) for i in range(len(c_raw))], 'y': c_raw.tolist()},
    'log': {'x': [float((e_log[i]+e_log[i+1])/2) for i in range(len(c_log))], 'y': c_log.tolist()},
}
with open('streamlit_data/eda/price_distribution.json', 'w') as f:
    json.dump(price_dist, f)

# 3. Summary stats
df.describe().round(2).to_parquet('streamlit_data/eda/summary_stats.parquet')

# 4. Estatísticas por bairro
neigh_stats = df.groupby('bairro')['preco_venda'].agg(['median', 'mean', 'count', 'std']).round(2)
neigh_stats['preco_m2_mediano'] = (
    df.assign(pm2=df['preco_venda'] / df['area_util'])
      .groupby('bairro')['pm2'].median().round(2)
)
neigh_stats.reset_index().to_parquet('streamlit_data/eda/neighborhood_stats.parquet')

# 5. Estatísticas por tipo
tipo_stats = df.groupby('tipo')['preco_venda'].agg(['median', 'mean', 'count']).round(2).reset_index()
tipo_stats.to_parquet('streamlit_data/eda/tipo_stats.parquet')

print('EDA: artefatos salvos.')

EDA: artefatos salvos.


In [4]:
# ── Bloco 3 · Feature Engineering ───────────────────────────────────────────
from sklearn.cluster import KMeans
from sklearn.neighbors import BallTree

df_feat = df.copy()

coords_rad = np.radians(df_feat[['lat', 'lon']].values)

# 1. Distância à praia mais próxima (feature-chave para o Rio!)
PRAIAS_REF = np.radians([
    [-22.9711, -43.1872],  # Copacabana
    [-22.9838, -43.2096],  # Ipanema
    [-22.9868, -43.2320],  # Leblon
    [-23.0004, -43.3649],  # Barra da Tijuca
    [-23.0186, -43.4710],  # Recreio
    [-22.9342, -43.1757],  # Aterro Flamengo
])
tree_praias = BallTree(PRAIAS_REF, metric='haversine')
dist_praia, _ = tree_praias.query(coords_rad, k=1)
df_feat['distancia_praia_km'] = (dist_praia[:, 0] * 6371).round(3)

# 2. Distância ao centro (Praça XV)
CENTRO = np.radians([[-22.9068, -43.1729]])
tree_centro = BallTree(CENTRO, metric='haversine')
dist_centro, _ = tree_centro.query(coords_rad, k=1)
df_feat['distancia_centro_km'] = (dist_centro[:, 0] * 6371).round(3)

# 3. Preço por m²
df_feat['preco_m2'] = (df_feat['preco_venda'] / df_feat['area_util']).round(2)

# 4. Ratios
df_feat['ratio_suites_quartos']    = (df_feat['suites']    / df_feat['quartos'].replace(0, 1)).round(3)
df_feat['ratio_banheiros_quartos'] = (df_feat['banheiros'] / df_feat['quartos'].replace(0, 1)).round(3)
df_feat['score_amenidades'] = (
    df_feat['varanda'] + df_feat['piscina'] +
    df_feat['academia'] + df_feat['portaria_24h'] + df_feat['vista_mar']
)

# 5. Transformações log
df_feat['log_area']       = np.log1p(df_feat['area_util'])
df_feat['log_condominio'] = np.log1p(df_feat['condominio'])

# 6. Clustering geográfico (K-Means k=10)
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
df_feat['geo_cluster'] = kmeans.fit_predict(df_feat[['lat', 'lon']])

pd.DataFrame(kmeans.cluster_centers_, columns=['lat', 'lon']).to_parquet(
    'streamlit_data/features/cluster_centers.parquet'
)

# 7. Pressão de vizinhança — versão full-dataset para visualização no mapa
# ⚠️  Esta versão usa todos os imóveis e NÃO é usada no modelo.
#     A versão leak-free (treino → treino/teste) é calculada no Bloco 4,
#     após o train/test split, e sobrescreve X_train e X_test.
tree_all = BallTree(coords_rad, metric='haversine')
_, nbr_idx = tree_all.query(coords_rad, k=51)
df_feat['pressao_vizinhanca'] = np.array([
    np.median(df_feat['preco_venda'].values[nbr_idx[i, 1:]])
    for i in range(len(df_feat))
]).round(2)

# 8. Target encoding (mediana do preço por bairro)
neigh_median = df_feat.groupby('bairro')['preco_venda'].median()
df_feat['bairro_mediana'] = df_feat['bairro'].map(neigh_median).round(2)

# 9. Features binárias derivadas
df_feat['novo_imovel']  = (df_feat['idade_imovel'] <= 3).astype(int)
df_feat['alto_andar']   = (df_feat['andar'] >= 10).astype(int)
df_feat['tipo_premium'] = df_feat['tipo'].isin(['Cobertura']).astype(int)

# 10. Remoção de outliers (percentis 0.5% e 99.5%)
q_lo = df_feat['preco_venda'].quantile(0.005)
q_hi = df_feat['preco_venda'].quantile(0.995)
n_before = len(df_feat)
df_feat = df_feat[(df_feat['preco_venda'] >= q_lo) & (df_feat['preco_venda'] <= q_hi)].copy()
print(f'Outliers removidos: {n_before - len(df_feat)} ({(n_before - len(df_feat))/n_before*100:.1f}%)')

# 11. Target log-transformado
df_feat['log_preco'] = np.log1p(df_feat['preco_venda'])

# 12. Matriz de correlação
NUM_COLS = [
    'preco_venda', 'area_util', 'quartos', 'suites', 'banheiros', 'vagas',
    'andar', 'idade_imovel', 'condominio', 'iptu',
    'distancia_praia_km', 'distancia_centro_km', 'pressao_vizinhanca', 'score_amenidades',
]
df_feat[NUM_COLS].corr().round(3).to_parquet('streamlit_data/features/correlation_matrix.parquet')

# Salva geo-features para mapa (usa pressao_vizinhanca full-dataset — só visualização)
df_feat[[
    'id_imovel', 'lat', 'lon', 'bairro', 'geo_cluster',
    'preco_venda', 'preco_m2', 'distancia_praia_km', 'pressao_vizinhanca'
]].to_parquet('streamlit_data/features/geo_features.parquet')

print(f'Feature engineering concluído. Shape: {df_feat.shape}')


Outliers removidos: 150 (1.0%)
Feature engineering concluído. Shape: (14850, 35)


In [5]:
# ── Bloco 4 · Modelagem ──────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neighbors import BallTree
import lightgbm as lgb

FEATURES = [
    'area_util', 'quartos', 'suites', 'banheiros', 'vagas', 'andar', 'idade_imovel',
    'varanda', 'piscina', 'academia', 'portaria_24h', 'vista_mar',
    'log_area', 'log_condominio',
    'distancia_praia_km', 'distancia_centro_km',
    'ratio_suites_quartos', 'ratio_banheiros_quartos', 'score_amenidades',
    'pressao_vizinhanca', 'bairro_mediana', 'geo_cluster',
    'novo_imovel', 'alto_andar', 'tipo_premium',
]
TARGET = 'log_preco'

df_model = df_feat[FEATURES + [TARGET, 'preco_venda', 'bairro']].dropna()
X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

# ── Recomputa pressao_vizinhanca sem data leakage ────────────────────────────
# BallTree fitado APENAS nos imóveis de treino.
# Treino: k=51, exclui o próprio imóvel (posição 0).
# Teste : k=50, todos os k vizinhos vêm do treino.
coords_train = np.radians(df_feat.loc[X_train.index, ['lat', 'lon']].values)
coords_test  = np.radians(df_feat.loc[X_test.index,  ['lat', 'lon']].values)
prices_train = df_feat.loc[X_train.index, 'preco_venda'].values

tree_pv = BallTree(coords_train, metric='haversine')

_, nbr_tr = tree_pv.query(coords_train, k=51)
pv_train = np.array([np.median(prices_train[nbr_tr[i, 1:]]) for i in range(len(coords_train))])

_, nbr_te = tree_pv.query(coords_test, k=50)
pv_test = np.array([np.median(prices_train[nbr_te[i]]) for i in range(len(coords_test))])

X_train = X_train.copy()
X_test  = X_test.copy()
X_train['pressao_vizinhanca'] = pv_train
X_test['pressao_vizinhanca']  = pv_test

print(f'pressao_vizinhanca (leak-free) — treino: {pv_train.mean():,.0f} | teste: {pv_test.mean():,.0f}')

# ── Ridge baseline ───────────────────────────────────────────────────────────
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train)
X_te_s = scaler.transform(X_test)
ridge = Ridge(alpha=10)
ridge.fit(X_tr_s, y_train)
y_pred_ridge = np.expm1(ridge.predict(X_te_s))

# ── LightGBM ─────────────────────────────────────────────────────────────────
lgb_params = {
    'objective': 'regression', 'metric': 'rmse',
    'num_leaves': 63, 'learning_rate': 0.05,
    'feature_fraction': 0.80, 'bagging_fraction': 0.80,
    'bagging_freq': 5, 'verbose': -1, 'seed': 42,
}
train_ds = lgb.Dataset(X_train, label=y_train, categorical_feature=['geo_cluster'])
val_ds   = lgb.Dataset(X_test,  label=y_test,  reference=train_ds)

lgbm_model = lgb.train(
    lgb_params, train_ds, num_boost_round=500,
    valid_sets=[val_ds],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)
y_pred_lgbm = np.expm1(lgbm_model.predict(X_test))
y_true = np.expm1(y_test.values)

def metrics(y_true, y_pred, name):
    mae  = mean_absolute_error(y_true, y_pred)
    mdae = float(np.median(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = float(np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1, None))) * 100)
    r2   = float(1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2))
    return {'modelo': name, 'MAE': round(mae, 0), 'MdAE': round(mdae, 0),
            'RMSE': round(rmse, 0), 'MAPE_%': round(mape, 2), 'R2': round(r2, 4)}

metrics_df = pd.DataFrame([metrics(y_true, y_pred_ridge, 'Ridge'),
                            metrics(y_true, y_pred_lgbm, 'LightGBM')])
metrics_df.to_parquet('streamlit_data/modeling/metrics.parquet')
print(metrics_df)

# Predições
pred_df = pd.DataFrame({
    'y_true':       y_true,
    'y_pred_ridge': y_pred_ridge,
    'y_pred_lgbm':  y_pred_lgbm,
    'bairro':       df_model.loc[X_test.index, 'bairro'].values,
})
pred_df.to_parquet('streamlit_data/modeling/predictions.parquet')

# Erro por faixa de preço
pred_df['faixa'] = pd.qcut(pred_df['y_true'], q=5, labels=['Q1','Q2','Q3','Q4','Q5'])
err_bin = pred_df.groupby('faixa', observed=True).apply(
    lambda x: pd.Series({
        'MAE_lgbm':  round(mean_absolute_error(x['y_true'], x['y_pred_lgbm']), 0),
        'MAE_ridge': round(mean_absolute_error(x['y_true'], x['y_pred_ridge']), 0),
        'n':         len(x),
    })
).reset_index()
err_bin.to_parquet('streamlit_data/modeling/error_by_bin.parquet')

# Erro por bairro
err_bairro = pred_df.groupby('bairro').apply(
    lambda x: pd.Series({
        'MAE_lgbm': round(mean_absolute_error(x['y_true'], x['y_pred_lgbm']), 0) if len(x) > 5 else np.nan,
        'n': len(x),
    })
).dropna().round(0).reset_index()
err_bairro.to_parquet('streamlit_data/modeling/error_by_bairro.parquet')

# Validação cruzada (5-fold) — também recomputa pressao_vizinhanca leak-free em cada fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
for fold, (tr_i, va_i) in enumerate(kf.split(X, y)):
    X_tr, X_va = X.iloc[tr_i].copy(), X.iloc[va_i].copy()
    y_tr, y_va = y.iloc[tr_i], y.iloc[va_i]

    c_tr = np.radians(df_feat.loc[X_tr.index, ['lat', 'lon']].values)
    c_va = np.radians(df_feat.loc[X_va.index, ['lat', 'lon']].values)
    p_tr = df_feat.loc[X_tr.index, 'preco_venda'].values
    t_cv = BallTree(c_tr, metric='haversine')
    _, ni_tr = t_cv.query(c_tr, k=51)
    _, ni_va = t_cv.query(c_va, k=50)
    X_tr['pressao_vizinhanca'] = [np.median(p_tr[ni_tr[i, 1:]]) for i in range(len(c_tr))]
    X_va['pressao_vizinhanca'] = [np.median(p_tr[ni_va[i]])     for i in range(len(c_va))]

    tr_d = lgb.Dataset(X_tr, label=y_tr, categorical_feature=['geo_cluster'])
    va_d = lgb.Dataset(X_va, label=y_va, reference=tr_d)
    m = lgb.train(lgb_params, tr_d, num_boost_round=lgbm_model.best_iteration,
                  valid_sets=[va_d], callbacks=[lgb.log_evaluation(-1)])
    p_va = np.expm1(m.predict(X_va))
    t_va = np.expm1(y_va.values)
    cv_rows.append({'fold': fold + 1,
                    'MAE':    round(mean_absolute_error(t_va, p_va), 0),
                    'MAPE_%': round(float(np.mean(np.abs((t_va - p_va) / np.clip(t_va, 1, None))) * 100), 2)})
pd.DataFrame(cv_rows).to_parquet('streamlit_data/modeling/cv_scores.parquet')

print('Modelagem concluída.')


Train: 11,880 | Test: 2,970
pressao_vizinhanca (leak-free) — treino: 1,016,708 | teste: 1,009,012
Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 0.133269
[200]	valid_0's rmse: 0.130999
Early stopping, best iteration is:
[209]	valid_0's rmse: 0.130933
     modelo       MAE     MdAE      RMSE  MAPE_%      R2
0     Ridge  142098.0  78427.0  243890.0   11.40  0.9581
1  LightGBM  134806.0  72091.0  238313.0   10.61  0.9599
Modelagem concluída.


In [6]:
# ── Bloco 5 · SHAP + Exportação ─────────────────────────────────────────────
import shap

X_shap = X_test.sample(min(1000, len(X_test)), random_state=42)
explainer   = shap.TreeExplainer(lgbm_model)
shap_values = explainer.shap_values(X_shap)

# Importância global (mean |SHAP|)
shap_imp = pd.DataFrame({
    'feature':        FEATURES,
    'mean_abs_shap':  np.abs(shap_values).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False)
shap_imp.to_parquet('streamlit_data/interpretation/shap_importance.parquet')

# Dependence plots (top 5 features por SHAP)
top5 = shap_imp['feature'].head(5).tolist()
dep_plots = {}
for feat in top5:
    fidx = FEATURES.index(feat)
    dep_plots[feat] = {
        'x':    X_shap[feat].values.tolist(),
        'shap': shap_values[:, fidx].tolist(),
    }
with open('streamlit_data/interpretation/shap_dependence.json', 'w') as f:
    json.dump(dep_plots, f)

# ── Simulação de impacto no preço ────────────────────────────────────────────
# Features binárias (toggle 0 → 1): mede o efeito de "ter" o atributo
BINARY_FEATS = {
    'vista_mar':    'Vista Mar',
    'piscina':      'Piscina',
    'portaria_24h': 'Portaria 24h',
    'varanda':      'Varanda',
    'academia':     'Academia',
    'novo_imovel':  'Imóvel Novo (≤3 anos)',
    'alto_andar':   'Alto Andar (≥10º)',
    'tipo_premium': 'Cobertura',
}

# Features de contagem (+1 unidade): mede o efeito de acrescentar uma unidade
COUNT_FEATS = {
    'quartos': '+1 Quarto',
    'suites':  '+1 Suíte',
    'vagas':   '+1 Vaga',
    'andar':   '+1 Andar',
}

n_sim = 200
sim_idx = np.random.default_rng(0).choice(len(X_train), n_sim, replace=False)
X_sim   = X_train.iloc[sim_idx].copy()
base_p  = np.expm1(lgbm_model.predict(X_sim))

impact = {}

for feat, label in BINARY_FEATS.items():
    X_on  = X_sim.copy(); X_on[feat]  = 1
    X_off = X_sim.copy(); X_off[feat] = 0
    delta = np.expm1(lgbm_model.predict(X_on)) - np.expm1(lgbm_model.predict(X_off))
    impact[label] = {
        'mean_delta': round(float(delta.mean()), 2),
        'pct_delta':  round(float((delta / np.clip(base_p, 1, None)).mean() * 100), 2),
        'tipo': 'binario',
    }

for feat, label in COUNT_FEATS.items():
    X_plus = X_sim.copy(); X_plus[feat] = X_sim[feat] + 1
    delta  = np.expm1(lgbm_model.predict(X_plus)) - base_p
    impact[label] = {
        'mean_delta': round(float(delta.mean()), 2),
        'pct_delta':  round(float((delta / np.clip(base_p, 1, None)).mean() * 100), 2),
        'tipo': 'contagem',
    }

with open('streamlit_data/interpretation/price_impact.json', 'w') as f:
    json.dump(impact, f)

# Salva modelo e lista de features
with open('streamlit_data/models/lgbm_imoveis_rj.pkl', 'wb') as f:
    pickle.dump(lgbm_model, f)
pd.Series(FEATURES, name='feature').to_frame().to_parquet('streamlit_data/models/features.parquet')

print('SHAP e exportação concluídos. Todos os artefatos em streamlit_data/.')
print(shap_imp.head(10).to_string(index=False))


SHAP e exportação concluídos. Todos os artefatos em streamlit_data/.
           feature  mean_abs_shap
         area_util       0.427278
    bairro_mediana       0.360899
      idade_imovel       0.058457
         vista_mar       0.055030
          log_area       0.048988
pressao_vizinhanca       0.033067
             andar       0.024379
            suites       0.019334
           quartos       0.018811
             vagas       0.016886
